In [1]:
import plotly
import numpy as np
import plotly.graph_objs as go

In [2]:
num_elements_x = 7
num_elements_y = 7

In [48]:
side_length = 0.4
u_min, u_max = -side_length/2, side_length/2
v_min, v_max = -side_length/2, side_length/2

# Resolution of the grid (higher = smoother)
num_u = 50
num_v = 50

u = np.linspace(u_min, u_max, num_u)
v = np.linspace(v_min, v_max, num_v)

goals_u = np.linspace(u_min, u_max, num_elements_x)
goals_v = np.linspace(v_min, v_max, num_elements_y)

U, V = np.meshgrid(u, v)
goals_U, goals_V = np.meshgrid(goals_u, goals_v)

A = 0.05
angle = np.pi
phase = np.pi/2
phase = np.pi/2
frequency = np.pi/0.4


# 2. Define your gamma(u, v)
def gamma_sur(u, v, A=0.05, base_pos=np.array([0, 0, 0])):
    x = u
    y = v
    s = u * np.cos(angle) + v * np.sin(angle)

    # z = A * np.sin(0.7*v + np.pi/6) * np.cos(u) # bump
    z = A * np.cos(frequency*s + phase) # bump

    # TODO modify x, y, and z such that (0, 0) lies at the base_pos
    offset = base_pos - np.array([0, 0, A * np.cos(frequency*(0) + phase)])
    #                                                          s = 0 at (0,0)
    x += offset[0]
    y += offset[1]
    z += offset[2]

    return x, y, z


goal_positions = dict()



X, Y, Z = gamma_sur(U, V, A=A)
goals_X, goals_Y, goals_Z = gamma_sur(goals_U, goals_V, A=A)

# 3. Make the Plotly surface figure
fig = go.Figure(data=[
    go.Surface(x=X, y=Y, z=Z),
    # go.Scatter3d(x=goals_X.flatten(), y=goals_Y.flatten(), z=goals_Z.flatten(), 
            # mode='markers', marker=dict(size=23, color='red')),
    # go.Scatter3d(x=[pos[0] for pos in link_positions],y=[pos[1] for pos in link_positions],z=[pos[2] for pos in link_positions],
    #         mode='markers', marker=dict(size=12, color='blue')),
])

fig.update_layout(
    title="The developable surface",
    scene=dict(
        aspectmode='manual',
        aspectratio=dict(x=1, y=1, z=0.5),
        xaxis_title="u",
        yaxis_title="v",
        zaxis_title="height"
    )
)

# 4. Show the interactive plotmass_mesh_open_tree
# fig.show()

plotly.io.write_html(fig, file="bump_surface.html", auto_open=True)

Opening in existing browser session.


In [ ]:
import plotly.io as pio


import numpy as np

# Parameters you can tune
distance = 3.0        # how far from center
yaw = np.deg2rad(75)  # rotation around z
pitch = np.deg2rad(20)  # tilt up/down

# Convert spherical → Cartesian
eye_x = distance * np.cos(pitch) * np.cos(yaw)
eye_y = distance * np.cos(pitch) * np.sin(yaw)
eye_z = distance * np.sin(pitch)


In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# ---------- your grid ----------
side_length = 0.4
u_min, u_max = -side_length/2, side_length/2
v_min, v_max = -side_length/2, side_length/2
num_u = 60
num_v = 60
u = np.linspace(u_min, u_max, num_u)
v = np.linspace(v_min, v_max, num_v)
U, V = np.meshgrid(u, v)

# ---------- surface params ----------
A = 0.05
frequency = np.pi/0.4

# choose your sweep
angles = np.deg2rad([0, 30, 60])          # columns (example)
phases = [0.0, np.pi/4, np.pi/2] # rows (example)

rows, cols = len(phases), len(angles)

# ---------- precompute z-range for consistent geometry scale ----------
Z_all = []
for ph in phases:
    for ang in angles:
        angle = ang
        phase = ph
        _, _, Z = gamma_sur(U, V)
        Z_all.append(Z)
zmin = min(Z.min() for Z in Z_all)
zmax = max(Z.max() for Z in Z_all)

# ---------- camera (shared) ----------
camera = dict(
    eye=dict(x=eye_x, y=eye_y, z=eye_z),
    center=dict(x=0.0, y=0.0, z=0.0),
    up=dict(x=0.0, y=0.0, z=1.0),
)

# ---------- make subplot figure (3D scenes) ----------
colorscale = "plasma"  # or any other Plotly colorscale
fig = make_subplots(
    rows=rows, cols=cols,
    specs=[[{"type": "scene"} for _ in range(cols)] for _ in range(rows)],
    subplot_titles=[
        f"phase={ph:.2f}, angle={np.rad2deg(ang):.0f}°"
        for ph in phases for ang in angles
    ],
    horizontal_spacing=0.02,
    vertical_spacing=0.05,
    aspectmode='manual',
    aspectratio=dict(x=1, y=1, z=0.5),


)

# Add surfaces
k = 0
for r, ph in enumerate(phases, start=1):
    for c, ang in enumerate(angles, start=1):
        angle = ang
        phase = ph
        X, Y, Z = gamma_sur(U, V)

        fig.add_trace(
            go.Surface(
                x=X, y=Y, z=Z,
                coloraxis="coloraxis",   # <- shared color scale across ALL subplots
                colorscale=colorscale,
                showscale=False          # <- hide per-trace scales
            ),
            row=r, col=c
        )
        k += 1

# Single shared colorbar + fixed color scale range (optional but usually what you want)
fig.update_layout(
    coloraxis=dict(
        cmin=zmin, cmax=zmax,   # shared scale
        colorbar=dict(title="z", len=0.85)
    ),
    width=1200,
    height=1400,
    margin=dict(l=10, r=10, t=80, b=10),
)

# Apply same camera + axis ranges to every scene
for i in range(1, rows*cols + 1):
    scene_key = "scene" if i == 1 else f"scene{i}"
    fig.layout[scene_key].update(
        camera=camera,
        xaxis=dict(range=[u_min, u_max], title="u"),
        yaxis=dict(range=[v_min, v_max], title="v"),
        zaxis=dict(range=[zmin, zmax], title="z"),
        aspectmode="cube",  # consistent scaling across scenes
    )

# Export one PNG
pio.write_image(fig, "grid_angle_phase.png", scale=2)
# Or interactive HTML:
pio.write_html(fig, "grid_angle_phase.html", auto_open=True)


TypeError: Figure.add_trace() got an unexpected keyword argument 'aspectmode'